# Ansys HFSS to AWS Palace: one-notebook pipeline

Run the cells **top to bottom**. At the end you have **one folder** containing
everything the HPC needs. Drag it over and run `qsub run_palace.pbs`.

**Before you start:** HFSS must be open with your project loaded, no solve
running, and this notebook must live in the pipeline folder (next to
`export_for_palace.py`, `mesh_any.py`, `palace_matchers.py`, `step_bodies.py`,
`write_palace_config.py`, `mesh_stats.py`, `hfss_size_report.py`,
`palace_pyepr.ipynb`, `palace_epr.py`).

**Restart the kernel when you switch to a different design.** The notebook
holds cached body lists and classifications; running a second design in the
same kernel can mix state from the first, and the result is not an error but
a mesh built from two designs.

| step | what happens | where |
|---|---|---|
| 1 | settings, the only cell you routinely edit | here |
| 2 | export geometry + physics from HFSS into a fresh run folder | your PC |
| 3 | review the mesh size for every body | your PC |
| 3b | read HFSS's own mesh operations, choose which to adopt | your PC |
| 4 | build the mesh (gmsh) + quality and budget report | your PC |
| 5 | generate `palace_config.json` | your PC |
| 6 | generate `run_palace.pbs` + completeness check | your PC |
| 7 | copy folder to HPC, `qsub` | HPC |
| 8 | copy `postpro_<TAG>/` back, record the run | your PC |
| 8b | open `palace_pyepr.ipynb` **inside the run folder**: pyEPR / χ analysis | your PC |

## 1. Settings

In [ ]:
# ============================== 1. SETTINGS ==============================
# The only cell you routinely edit.

DESIGN_NAME = "Vostok-3"   # HFSS design to export (None = active design)
TAG         = "run1"       # label stamped into mesh/config filenames

# --- mesh sizes (mm) you want to FORCE. These beat every other tier,
# including anything adopted from HFSS in step 3b. Leave empty {} to let
# the design's own mesh operations and the analytic fallbacks decide.
# Consult mesh_size_database.json for the lab's tested values.
#   0.0002 = 0.2 um, 0.05 = 50 um, 1.0 = 1 mm
SIZES = {
    # "JJ": 0.0002,
    # "thin_lead": 0.0005,
    # "pads": 0.15,
}

# --- the .aedt project file, for step 3b. None = auto-find the newest
# .aedt next to the run folder. Point it at an archived project to read
# an older variant's mesh operations instead. NOTE: the file is NOT
# named after the design inside it, and one file can hold many designs.
AEDT_PATH = None    # e.g. r"C:\qcrew\designs\test_export.aedt"

# --- global coarsening knob. Multiplies EVERY body size uniformly
# (junction bodies and explicit SIZES entries are exempt). PROVEN
# RECIPE = 1.0: coarsening changes the discretization and is what broke
# Vostok-3 (chords of nearby curves crossing -> PLC errors). Only raise
# this on a design that has already meshed clean, one step at a time,
# watching the conformity check.
#
# Note SCALE saturates on µm-graded designs: 1.0 -> 3.0 removed only 36%
# of tets on Vostok-3, because the element count lives in the grading
# bands around µm-wide features, and all grading distances scale with
# the size. Per-body sizes are the real coarsening lever, not SCALE.
SCALE = 1.0

# --- mesher options
# PROVEN RECIPE (Vostok-2 run2, job 1314089: meshed, solved, 2 AMR
# passes, exit 0): order 2, curvature 24, delaunay, no scale, no fuzz.
# The safe speedups below (threads, no vtk, consolidated grading fields,
# meshing on LOCAL disk not \\qcrew_drive) cut the 7-hour mesh time
# WITHOUT changing the discretization.
MESH_ORDER = 2        # 2 = curvilinear (proven)
CURVATURE  = 24       # proven value. 12 halves curve resolution, which
                      # is a DISCRETIZATION change; do not bundle it
                      # with speed fixes
MESH_ALGO3D  = "delaunay"  # proven kernel. "hxt" is faster but packs
                           # ~2x denser and changes the discretization
MESH_THREADS = 0      # 0 = all cores (safe speedup: same mesh recipe)
WRITE_VTK    = False  # safe speedup: skip the huge ASCII ParaView copy
BOOLEAN_TOL  = 0      # LEAVE AT 0. The 1e-4 experiment produced an
                      # all-skin-no-flesh mesh (1252 tets, 270k
                      # triangles) that MFEM aborted on. Fuzzy booleans
                      # are a last resort for PLC errors, gated by the
                      # conformity check, never a default.

# --- Palace solver options (written into the config)
PALACE = {
    "target_freq_GHz": 3.5,   # keep BELOW the lowest expected mode
    "n_modes": 5,
    "solver_order": 1,        # the proven AMR profile solved at p=1
                              # (4.49M unknowns on 3.44M tets); p=2 on
                              # this same mesh = 23M unknowns = OOM.
                              # p=1 frequencies wander a few hundred MHz
                              # between passes: trust the mode COUNT and
                              # BAND, not the numbers
    "amr_max_its": 5,         # let update_fraction decide how many
                              # passes actually fit under max_size
    "amr_update_fraction": 0.25,
                              # THE 3-PASS LEVER, config-only, zero mesh
                              # risk: the proven run (higher UF) grew
                              # +10% then +51% and hit the cap after 2
                              # passes. Smaller UF = smaller passes =
                              # the same growth budget over 3-4
    "amr_max_size": 7000000,  # unknowns cap. 7M = the 2xA40 VRAM
                              # ceiling. On a 240 GB CPU node this is an
                              # artificial limit inherited from the GPU
                              # build: raise it incrementally there
    "device": "gpu",          # "gpu" = validated iterative A40 profile,
                              #         capped ~8M unknowns by VRAM
                              # "cpu" = CPU profile, 240 GB, use this
                              #         when you need real resolution
}

# --- HPC job parameters (for run_palace.pbs). The CPU-vs-GPU choice
# comes from PALACE["device"] above, so config and PBS always agree.
HPC = {
    "jobname": f"palace_{TAG}",
    "walltime": "06:00:00",
    "ncpus": 36,       # CPU jobs; GPU jobs auto-clamp to 8 cores
    "mem_gb": 240,     # host RAM; GPU queue default is 240gb
    "palace_bin": None,   # None = auto-pick the right build
                          # (openmpi for cpu, cuda for gpu)
}

import os
import pipeline_helpers as ph
import pipeline_state as ps

NOTEBOOK_DIR = os.getcwd()
print("settings loaded; notebook folder:", NOTEBOOK_DIR)

## 1b. Resume a previous run (skip on a fresh run)

Everything the pipeline makes after the export lives on disk. What a kernel
restart destroys is only the Python names pointing at those files, and
`RUN_DIR` above all: recovering it any other way means re-running the
exporter, which needs HFSS open and creates a new timestamped folder you did
not ask for.

Run **this cell instead of step 2** to pick up where you left off. It reads
`notebook_state.json` from the run folder and checks each recorded file is
still on disk, rather than trusting the record.

Leave `RESUME_DIR = None` to reopen the most recent run from this notebook
folder, or set it to a specific path to go back to an older one.

In [ ]:
RESUME_DIR = None      # None = the newest run saved from this folder

_state = ps.resume(RESUME_DIR, notebook_dir=NOTEBOOK_DIR)

if _state:
    RUN_DIR = _state["run_dir"]
    TAG     = _state.get("tag", TAG)
    SIZES   = _state.get("sizes", SIZES)
    MSH     = _state.get("mesh")
    GROUPS  = _state.get("groups")
    PALACE_CONFIG = _state.get("config")
    print(f"\nRUN_DIR = {RUN_DIR}")
    print("Skip step 2 and step 3b; continue from whichever step you "
          "were changing.")

## 2. Export from HFSS
Creates a fresh timestamped run folder next to your `.aedt` file with
`device.step`, `device_config.json`, and copies of the pipeline scripts.
HFSS stays open and nothing in your project is modified.

The export hard-fails rather than writing a partial config if any model body
has no physics role, if a junction has an unparseable inductance, or if the
active design is not the one named in `DESIGN_NAME`. A config with silently
missing physics is worse than no config.

In [ ]:
import shutil
import export_for_palace as exp

RUN_DIR = exp.main(
    design_name_arg=DESIGN_NAME,
    mesh_size_overrides={},        # sizes are applied at MESH time (step 4),
                                   # so the exported config stays pristine
    palace_solver_overrides=PALACE,
    mesher_overrides={"mesh_order": MESH_ORDER,
                      "curvature_elements_per_2pi": CURVATURE},
)
print("\nRUN_DIR =", RUN_DIR)

# Stamp the post-run EPR analysis into the run folder: when postpro_<TAG>
# comes back from vanda, the pyEPR notebook is already sitting next to it
# (step 8b). palace_epr.py is its engine -- pyEPR's real QuantumAnalysis
# fed from Palace CSVs. Same refresh-from-masters rule as every other
# stamped script.
for _fname in ("palace_pyepr.ipynb", "palace_epr.py"):
    _src = os.path.join(NOTEBOOK_DIR, _fname)
    if os.path.isfile(_src):
        shutil.copy2(_src, os.path.join(RUN_DIR, _fname))
        print(f"copied:  {_fname}")
    else:
        print(f"WARNING: {_fname} not found next to this notebook; "
              f"the run folder gets no EPR analysis")

ps.save(RUN_DIR, notebook_dir=NOTEBOOK_DIR, tag=TAG, design=DESIGN_NAME,
        mesh=None, groups=None, config=None)   # a new export invalidates
                                               # any mesh/config recorded
                                               # under an earlier run

## 3. Review mesh sizes
One line per body: the size the mesh will use and where it came from
(`operation` = set in HFSS, `stats` = HFSS's adapted mesh, `auto` = analytic
fallback). Lab-database hints appear where a body name matches a known
component.

The analytic fallbacks, for reference:

- sheets: narrowest in-plane dimension / 6
- dielectric solids: `min(lambda_material / 12, mid-dimension / 2)`
- conductor and PEC solids: smallest dimension / 3

**If anything looks wrong, put the correction in `SIZES` in cell 1 and re-run
from this cell.** No need to re-export.

In [ ]:
base_sizes = ph.size_report(
    RUN_DIR,
    database_path=os.path.join(NOTEBOOK_DIR, "mesh_size_database.json"))

if SIZES:
    print("\nyour overrides for this run:")
    for body, size in SIZES.items():
        print(f"  {body:<24}{size:g} mm")

## 3b. Read HFSS's own mesh operations

The `.aedt` project file is plain text, so this reads every enabled
length-based mesh operation straight out of it: no Ansys session, no licence
checkout, and it works on archived project files. The report prints each
operation with the expression exactly as typed in HFSS, its value in mm, and
the bodies it covers, then a ready-made `SIZES` block.

**Read the table before adopting anything.** HFSS length operations are
refinement *caps* from an adaptive solve that ran on a large CPU box with no
memory ceiling. Most of them are far finer than this pipeline's analytic
fallbacks, so adopting them all makes the mesh **larger**, not smaller. On
Vostok-1, 22 of 24 operations are below 65 µm and one applies 8.2 µm to 27
separate gap bodies.

Adopting an HFSS size is a coarsening win only where the operation is
**coarser** than the fallback. On Vostok-1 that is `GND` at 2.5 mm and
`Cavity_pin` at 1.3 mm. Everything else is provenance: useful for knowing
what HFSS did, not for making the mesh fit the budget.

Bodies with **no** HFSS operation are listed separately. The cavity is
always among them, because in HFSS it is background vacuum sized from
wavelength during the adaptive solve, so there is no length anywhere in the
file to read. Set those yourself or leave them to the fallback.

In [ ]:
# ---- read the mesh operations out of the .aedt ----------------------
import glob
import importlib
import hfss_size_report as hs
importlib.reload(hs)          # pick up edits without restarting the kernel

# Resolve the project file. AEDT_PATH = None means "newest .aedt next to
# the run folder", which is where export_for_palace.py put RUN_DIR.
if AEDT_PATH:
    aedt_file = AEDT_PATH
else:
    candidates = sorted(
        glob.glob(os.path.join(os.path.dirname(RUN_DIR), "*.aedt")),
        key=os.path.getmtime)
    if not candidates:
        raise FileNotFoundError(
            f"no .aedt found next to {os.path.dirname(RUN_DIR)}; "
            f"set AEDT_PATH in cell 1")
    aedt_file = candidates[-1]
    if len(candidates) > 1:
        print(f"{len(candidates)} .aedt files found; using the newest:")
print(f"reading: {aedt_file} "
      f"({os.path.getsize(aedt_file) / 1e6:.1f} MB)\n")

# Which designs does this file hold? One project can contain many, and
# the file name tells you nothing about them.
for name, data in hs.scan(aedt_file).items():
    marker = "  <-- DESIGN_NAME" if name == DESIGN_NAME else ""
    print(f"  {name:<40} {len(data['ops']):>3} op(s), "
          f"{len(data['vars']):>4} var(s){marker}")
print()

hfss_sizes = hs.sizes_from_aedt(
    aedt_file,
    design=DESIGN_NAME,
    config_path=os.path.join(RUN_DIR, "device_config.json"),
)

### Adopt the sizes you want

Paste the entries you want from the block printed above into `HFSS_SIZES`
below, then add anything HFSS never sized (the cavity is the usual case).
Leave `HFSS_SIZES` empty to ignore the HFSS operations entirely and let the
export's own tiers decide.

Precedence after this cell: your `SIZES` from cell 1 wins over `HFSS_SIZES`,
which wins over the export's operation, stats, and fallback tiers.

Everything set here is an absolute length, so `SCALE` does not touch it,
exactly like a value typed into Ansys. That is deliberate, and it means a
long `HFSS_SIZES` list disables most of your coarsening.

In [ ]:
# ---- paste the entries you want to adopt ----------------------------
HFSS_SIZES = {
    # e.g. the two operations that are COARSER than the fallbacks:
    # "GND":        2.5,
    # "Cavity_pin": 1.3,
}

# ---- bodies HFSS never sized: set them yourself ---------------------
# The cavity has no mesh operation in any HFSS design, because it is
# background vacuum sized from wavelength during the adaptive solve.
# Leave commented to accept the mesher's lambda/12 fallback.
MANUAL_SIZES = {
    # "cavity": 2.0,
}

# ---- combine, explicit SIZES from cell 1 winning --------------------
_before = dict(SIZES)
SIZES = {**HFSS_SIZES, **MANUAL_SIZES, **_before}

print(f"{len(HFSS_SIZES)} adopted from HFSS, {len(MANUAL_SIZES)} manual, "
      f"{len(_before)} from cell 1 (these win)")
if SIZES:
    print("\nsizes going into the mesher:")
    for body in sorted(SIZES, key=lambda b: SIZES[b]):
        if body in _before:
            source = "cell 1"
        elif body in MANUAL_SIZES:
            source = "manual"
        else:
            source = "HFSS op"
        print(f"  {body:<28}{SIZES[body]:<10g}({source})")
else:
    print("\nno explicit sizes: every body uses the export's own tiers")

ps.save(RUN_DIR, notebook_dir=NOTEBOOK_DIR, sizes=SIZES)

## 4. Mesh
Builds `device_<TAG>.msh` with your sizes, then prints the quality report,
the conformity check, and the **budget block**.

Read the budget block before doing anything else. It converts the tet count
into unknowns at Order 1 and Order 2 and tells you how many AMR passes fit
under `amr_max_size`. Both historical OOM crashes were predictable from the
printed tet count alone. If your Order's verdict is not "GOOD for >=3 its",
change sizes and re-run **this cell**; no re-export needed.

Rough targets under the 7M-unknown cap with small-UF passes: about 1.58M
tets at Order 1, about 305k tets at Order 2.

The conformity check raises if any surface triangle is not a face of a
tetrahedron, which is exactly the condition that makes Palace abort inside
MFEM with an unreadable `STable3D` error. Better to catch it here in seconds
than on the cluster after a queue wait.

In [ ]:
MSH, GROUPS = ph.run_mesher(RUN_DIR, TAG, sizes=SIZES,
                            mesh_order=MESH_ORDER,
                            curvature_segments=CURVATURE,
                            scale=SCALE,
                            threads=MESH_THREADS,
                            algo3d=MESH_ALGO3D,
                            vtk=WRITE_VTK,
                            boolean_tol=BOOLEAN_TOL)
ph.run_mesh_stats(RUN_DIR, MSH)
ph.run_conformity_check(RUN_DIR, MSH)   # raises if Palace would abort
                                        # (MFEM STable3D) on this mesh

ps.save(RUN_DIR, notebook_dir=NOTEBOOK_DIR, sizes=SIZES,
        mesh=os.path.basename(str(MSH)),
        groups=os.path.basename(str(GROUPS)),
        config=None)   # a new mesh invalidates the previous config

## 5. Palace config
Generated entirely from the groups file and `device_config.json`: materials,
boundaries, junction port, solver block. Read the printed summary and check
it names every physical group you expect.

`apply_solver_overrides` refreshes the solver settings from this notebook's
`PALACE` dict first, so editing `PALACE` and re-running this cell works
without re-exporting. **The config must be regenerated for every run.** A
stale config carrying the wrong device target or mesh path will either crash
or, worse, produce plausible wrong results.

In [ ]:
ph.apply_solver_overrides(RUN_DIR, PALACE)
PALACE_CONFIG = ph.run_writer(RUN_DIR, GROUPS,
                              out_config=f"palace_config_{TAG}.json",
                              mesh=MSH,
                              postpro_dir=f"postpro_{TAG}",
                              device=PALACE.get("device", "cpu"))

ps.save(RUN_DIR, notebook_dir=NOTEBOOK_DIR,
        config=os.path.basename(str(PALACE_CONFIG)),
        palace=PALACE)

## 6. PBS file + completeness check
Writes `run_palace.pbs` (with the crashed-run `postpro` guard built in) and
verifies the folder is internally consistent: mesh, config, and pbs all
reference each other.

In [ ]:
USE_GPU = PALACE.get("device", "cpu") == "gpu"
print(f"PBS mode: {'GPU (2xA40, free gpu queue)' if USE_GPU else 'CPU'}")
ph.write_pbs(RUN_DIR, PALACE_CONFIG, postpro_dir=f"postpro_{TAG}",
             gpu=USE_GPU, **HPC)
ph.final_checklist(RUN_DIR, MSH, PALACE_CONFIG)

## 7. On the HPC

Copy the whole run folder to the cluster (drag it in your file browser, or
`scp -r`). Then:

```bash
cd <the folder>
qsub run_palace.pbs
```

Watch the job with `qstat -u $USER`. The mode table appears in the job's
`.o` log file, and machine-readable results in `postpro_<TAG>/eig.csv`.

**Check the process count in the Palace banner first if anything looks
strange.** A build run against the wrong MPI can start as N independent
single-rank copies of the whole problem rather than one N-rank job, which
looks like a memory or performance problem and is neither.

**Afterwards:** for field plots, add `"Save": 5` inside `Solver.Eigenmode`
of the palace config before submitting, and pull back
`postpro_<TAG>/paraview/`. For a mesh sweep, change `TAG` and `SIZES` in
cell 1 and re-run from step 3; each tag gets its own mesh and config.

## 8. After the run: record it
Copy the job's `.o` log **and/or the `postpro_<TAG>/` folder** back into the
run folder (a `.png` screenshot of the design too, if you have one). This
cell parses everything available: modes per pass, tets total and per
component, AMR iterations, wall time, memory, how the run ended. It appends
a record to the lab registry and regenerates `runlog.html`.

The log gives per-pass history, which is what you need for the Δf-per-pass
convergence record. `eig.csv` alone gives only the final modes. Either
works, both is best.

## 8b. Quantum analysis (pyEPR)

The export stamped `palace_pyepr.ipynb` + `palace_epr.py` into the run
folder. Once `postpro_<TAG>/` is back from vanda, open
**`palace_pyepr.ipynb` inside the run folder** (not this notebook) and run
it top to bottom in the `qiskit_metal` env. It auto-finds the postpro
folder (highest AMR iteration), reads the junction L/C from the palace
config the solve actually used, and runs pyEPR's real `QuantumAnalysis`:
χ matrix, anharmonicities, dressed frequencies, g couplings — the same
code path as the Ansys `EPR_master_nb`. Results land as
`epr_*_<TAG>.csv` next to the mesh, so they travel with the run.

In [ ]:
LOG_FILE = "palace_eigen.o1234567"   # PBS .o file in RUN_DIR, or None
NOTES    = "which sizes / what this run tests"
REGISTRY = os.path.join(os.path.dirname(RUN_DIR), "palace_runs")

import subprocess, sys
cmd = [sys.executable, os.path.join(RUN_DIR, "palace_runlog.py"), "add",
       "--run-dir", RUN_DIR, "--registry", REGISTRY, "--notes", NOTES]
if LOG_FILE:
    cmd += ["--log", os.path.join(RUN_DIR, LOG_FILE)]
# postpro_<TAG>/ is auto-detected from the config if you copied it back
subprocess.run(cmd, check=True)
print("open", os.path.join(REGISTRY, "runlog.html"))